In [3]:
from pathlib import Path
import numpy as np

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe


# ============================================================
# Paths
# ============================================================

PROJECT_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn")

DATA_PATH = PROJECT_DIR / "datasets" / "ECG5000" / "ECG5000_val_split.txt"

# 原始/clean QONNX，作为 float target
TARGET_QONNX_PATH = PROJECT_DIR / "onnx_exports" / "ecg5000_pointwise_qtcn_qonnx_clean_shaped_fixed.onnx"

# FINN model，推荐用 step_convert_to_hw，因为它还保留最后 Mul/Add，输出更接近 float logits
FINN_MODEL_PATH = PROJECT_DIR / "finn_build_pynqz2" / "intermediate_models" / "step_specialize_layers.onnx"

NUM_CLASSES = 5
SEQ_LEN = 140
MAX_SAMPLES = 100

# 你要测试的 input scale
CANDIDATE_SCALES = [1.0, 2.0, 4.0, 8.0, 16.0, 32.0]


# ============================================================
# Helpers
# ============================================================

def safe_bincount(x, minlength=5):
    x = np.asarray(x).astype(np.int32).reshape(-1)
    return np.bincount(x, minlength=minlength)


def logits_to_2d_np(x):
    x = np.asarray(x)

    if x.ndim == 1 and x.size == NUM_CLASSES:
        return x.reshape(1, NUM_CLASSES)

    if x.ndim == 2:
        return x

    if x.ndim == 4:
        # [N, 5, 1, 1]
        if x.shape[1] == NUM_CLASSES and x.shape[2] == 1 and x.shape[3] == 1:
            return x[:, :, 0, 0]

        # [N, 1, 1, 5] or [N, L, 1, 5]
        if x.shape[-1] == NUM_CLASSES and x.shape[2] == 1:
            return x[:, -1, 0, :]

        # [N, 5, L, 1]
        if x.shape[1] == NUM_CLASSES and x.shape[3] == 1:
            return x[:, :, -1, 0]

        # [N, 1, L, 5]
        if x.shape[-1] == NUM_CLASSES and x.shape[1] == 1:
            return x[:, 0, -1, :]

    raise ValueError(f"Cannot convert output shape {x.shape} to [N, 5]")


def pick_logits_output(out_dict):
    for name, val in out_dict.items():
        try:
            logits = logits_to_2d_np(val)
            if logits.shape[1] == NUM_CLASSES:
                return name, logits.astype(np.float32)
        except Exception:
            pass

    print("Available outputs:")
    for name, val in out_dict.items():
        print(name, np.asarray(val).shape)

    raise RuntimeError("No logits-like output found.")


def reshape_for_model(x_1d, input_shape):
    input_shape = tuple(int(v) for v in input_shape)

    if int(np.prod(input_shape)) != SEQ_LEN:
        raise ValueError(f"Input shape {input_shape} does not contain {SEQ_LEN} values")

    return x_1d.reshape(input_shape).astype(np.float32)


def run_model_samplewise(model, x_values, mode_name):
    input_name = model.graph.input[0].name
    input_shape = model.get_tensor_shape(input_name)

    logits_list = []

    for i in range(x_values.shape[0]):
        x = reshape_for_model(x_values[i], input_shape)
        out_dict = oxe.execute_onnx(model, {input_name: x})

        if i == 0:
            print(f"\n=== First output for {mode_name} ===")
            for name, val in out_dict.items():
                print(name, np.asarray(val).shape, np.asarray(val).reshape(-1)[:10])

        _, logits = pick_logits_output(out_dict)
        logits_list.append(logits.reshape(1, NUM_CLASSES))

        if (i + 1) % 50 == 0 or (i + 1) == x_values.shape[0]:
            print(f"[{mode_name}] executed {i + 1}/{x_values.shape[0]}")

    return np.concatenate(logits_list, axis=0)


def diff_stats(a, b):
    d = np.abs(np.asarray(a, dtype=np.float32) - np.asarray(b, dtype=np.float32))
    return {
        "max": float(np.max(d)),
        "mean": float(np.mean(d)),
        "median": float(np.median(d)),
        "p95": float(np.percentile(d, 95)),
    }


# ============================================================
# Main
# ============================================================

def main():
    print("===================================================")
    print("Check whether FINN input needs scaling")
    print("===================================================")
    print("DATA_PATH        :", DATA_PATH)
    print("TARGET_QONNX_PATH:", TARGET_QONNX_PATH)
    print("FINN_MODEL_PATH  :", FINN_MODEL_PATH)

    data = np.loadtxt(str(DATA_PATH), dtype=np.float32)
    labels_raw = data[:, 0].astype(np.int32)
    x_raw = data[:, 1:].astype(np.float32)

    # ECG5000 labels 1..5 -> 0..4
    if sorted(np.unique(labels_raw).tolist()) == [1, 2, 3, 4, 5]:
        labels = labels_raw - 1
    else:
        labels = labels_raw

    if MAX_SAMPLES is not None:
        x_raw = x_raw[:MAX_SAMPLES]
        labels = labels[:MAX_SAMPLES]

    print("Samples:", len(labels))
    print("Label distribution:", safe_bincount(labels, NUM_CLASSES))
    print("Raw first 10:", x_raw[0, :10])
    print("round(raw) first 10:", np.round(x_raw[0, :10]))
    print("round(raw*16) first 10:", np.round(x_raw[0, :10] * 16.0))

    target_model = ModelWrapper(str(TARGET_QONNX_PATH))
    finn_model = ModelWrapper(str(FINN_MODEL_PATH))

    target_input = target_model.graph.input[0].name
    finn_input = finn_model.graph.input[0].name

    print("\nTarget input:")
    print(" name :", target_input)
    print(" shape:", target_model.get_tensor_shape(target_input))
    try:
        print(" dtype:", target_model.get_tensor_datatype(target_input))
    except Exception as e:
        print(" dtype: <error>", e)

    print("\nFINN input:")
    print(" name :", finn_input)
    print(" shape:", finn_model.get_tensor_shape(finn_input))
    try:
        print(" dtype:", finn_model.get_tensor_datatype(finn_input))
    except Exception as e:
        print(" dtype: <error>", e)

    # Target QONNX uses raw float model input
    target_logits = run_model_samplewise(target_model, x_raw, "target_qonnx_raw")
    target_preds = target_logits.argmax(axis=1)

    print("\n===================================================")
    print("Target QONNX result")
    print("===================================================")
    print("Accuracy:", np.mean(target_preds == labels))
    print("Prediction distribution:", safe_bincount(target_preds, NUM_CLASSES))
    print("First target logits:", target_logits[0])
    print("First target pred:", int(target_preds[0]), "label:", int(labels[0]))

    print("\n===================================================")
    print("Candidate FINN input scales")
    print("===================================================")

    best = None

    for scale in CANDIDATE_SCALES:
        print("\n---------------------------------------------------")
        print(f"Testing FINN input = round(raw * {scale})")
        print("---------------------------------------------------")

        x_int = np.round(x_raw * scale)

        # global_in is INT8, so clip to INT8 range
        x_int = np.clip(x_int, -128, 127).astype(np.float32)

        print("First input first 10:", x_int[0, :10])

        finn_logits = run_model_samplewise(
            finn_model,
            x_int,
            f"finn_round_x{scale}",
        )
        finn_preds = finn_logits.argmax(axis=1)

        acc = float(np.mean(finn_preds == labels))
        agree = float(np.mean(finn_preds == target_preds))
        st = diff_stats(finn_logits, target_logits)

        print(f"\nScale {scale} result:")
        print("  accuracy vs labels       :", acc)
        print("  pred agreement with QONNX:", agree)
        print("  pred distribution        :", safe_bincount(finn_preds, NUM_CLASSES))
        print(
            "  logits diff vs QONNX     : "
            f"max={st['max']:.6g}, mean={st['mean']:.6g}, "
            f"median={st['median']:.6g}, p95={st['p95']:.6g}"
        )
        print("  first FINN logits        :", finn_logits[0])
        print("  first FINN pred          :", int(finn_preds[0]))

        score = agree
        if best is None or score > best["agree"]:
            best = {
                "scale": scale,
                "acc": acc,
                "agree": agree,
                "diff": st,
            }

    print("\n===================================================")
    print("Decision")
    print("===================================================")
    print("Best scale by prediction agreement with QONNX:")
    print(best)

    print("\nRecommended PYNQ setting:")
    if best["scale"] == 1.0:
        print('INPUT_PREPROCESS_MODE = "round"')
        print("# send round(raw)")
    else:
        print(f"INPUT_SCALE_FROM_BREVITAS = {best['scale']}")
        print(f"# send round(raw * {best['scale']})")

    print("\nImportant:")
    print("Do not use raw float directly when FINN input dtype is INT8.")
    print("The input must be integer-valued.")


if __name__ == "__main__":
    main()

ECG5000 FPGA input-scale sweep
[info] DEPLOY_ROOT: /home/xilinx/jupyter_notebooks/Thesis/deploy
[info] BITFILE_DIR: /home/xilinx/jupyter_notebooks/Thesis/deploy/bitfile
[info] DRIVER_DIR : /home/xilinx/jupyter_notebooks/Thesis/deploy/driver
[info] DRIVER_PY  : /home/xilinx/jupyter_notebooks/Thesis/deploy/driver/driver.py
[info] TEST_TXT   : /home/xilinx/jupyter_notebooks/Thesis/data/ECG5000_val_split.txt
[info] OUT_ROOT   : /home/xilinx/jupyter_notebooks/Thesis/results_input_scale_sweep
[info] REQUESTED_BATCH_SIZE: 1
[info] USE_PREPACKED_INPUTS: False
[info] APPLY_OUTPUT_AFFINE: False
[info] INPUT_PREPROCESS_MODES: ['round', 'round_x16']


PermissionError: [Errno 13] Permission denied: '/home/xilinx'